# BeautifulSoup vs XPath: How to Select HTML Elements

**From the simplest to the most complex — every case is shown TWO ways:
BeautifulSoup and XPath.**

The goal: once you understand the *concept of selecting elements*, you can use either tool.

**Who it's for:** participants who already know basic HTML & want to get good at selecting elements.

**What we'll cover:** selecting by **tag**, **class** (including multi-class),
**id**, **attributes** (`data-*`, `aria-label`, `role`, `href`), **partial** matching
(`contains`, `starts-with`, regex), **text** matching, **parent/sibling/child** navigation,
all the way to **combined conditions** and **numeric** filters.

> **Important note:** BeautifulSoup does **not** support XPath. For XPath on static HTML
> we use **`lxml`** (`tree.xpath(...)`). lxml is already a dependency of this project.
> (In Selenium, XPath is used via `By.XPATH` — see the `walkthrough.ipynb` notebook.)


## Outline (simple → complex)

0. Setup — sample HTML + the `soup` (BS4) & `tree` (lxml) objects
1. Select by **tag**
2. Select by **id**
3. Select by **class** (1 token)
4. **Multi-class** — an important difference between BS4 & XPath ⚠️
5. Elements **without a class** → use **position** (nth)
6. Select by **attribute** (`data-*`)
7. **aria-label** & **role** (accessibility attributes)
8. **href** — get all links, internal links, mailto
9. **Partial** matching — `contains`, `starts-with`, regex
10. Select by **text**
11. Navigation: **parent / ancestor**
12. Navigation: **sibling**
13. Navigation: **direct child** (`/` vs `//`)
14. **Combined** conditions + **numeric** filters
15. Mini-project: extract all products → list of dicts (BS4 vs XPath)
16. Cheat sheet, pitfalls, exercises


In [1]:
import re

from bs4 import BeautifulSoup
from lxml import html as lxml_html

# Sample HTML: similar to an e-commerce catalog page, intentionally varied:
# - single class & multi-class           - has an id
# - has data-*, aria-label, role attrs   - has internal/external/mailto hrefs
# - has elements WITHOUT a class          - has a product with incomplete data
HTML = """
<html>
  <body>
    <header id="top-nav" role="navigation" aria-label="Main menu">
      <a href="/" class="logo">OurShop</a>
      <ul>
        <li><a href="/category/books">Books</a></li>
        <li><a href="/category/electronics">Electronics</a></li>
        <li><a href="https://promo.external.com/diskon">External Promo</a></li>
      </ul>
    </header>

    <main id="content">
      <h1>Product List</h1>
      <p>Showing a selection of featured products.</p>

      <div class="product-list">
        <div class="product card featured" data-id="101" data-category="books">
          <h2 class="product-name">Learn Python</h2>
          <span class="price" data-price="75000">Rp75.000</span>
          <span class="rating" aria-label="Rating 4.5 out of 5">4.5</span>
          <a href="/product/101" class="btn btn-primary" aria-label="View Learn Python">View</a>
          <button aria-label="Add Learn Python to cart">Add</button>
        </div>

        <div class="product card" data-id="102" data-category="electronics">
          <h2 class="product-name">Wireless Mouse</h2>
          <span class="price" data-price="150000">Rp150.000</span>
          <span class="rating" aria-label="Rating 4.0 out of 5">4.0</span>
          <a href="/product/102" class="btn btn-primary" aria-label="View Wireless Mouse">View</a>
          <button aria-label="Add Wireless Mouse to cart">Add</button>
        </div>

        <div class="product card" data-id="103" data-category="books">
          <h2 class="product-name">Data Engineering 101</h2>
          <span class="price" data-price="120000">Rp120.000</span>
          <span class="rating" aria-label="Rating 5.0 out of 5">5.0</span>
          <a href="/product/103" class="btn btn-primary" aria-label="View Data Engineering 101">View</a>
          <button aria-label="Add Data Engineering 101 to cart">Add</button>
        </div>

        <div class="product card" data-id="104" data-category="electronics">
          <h2 class="product-name">Mechanical Keyboard</h2>
          <span class="price" data-price="350000">Rp350.000</span>
          <span class="badge">Out of stock</span>
          <a href="/product/104" class="btn btn-disabled" aria-label="View Mechanical Keyboard">View</a>
        </div>
      </div>

      <footer>
        <p>Contact: <a href="mailto:cs@tokokita.id">cs@tokokita.id</a></p>
        <span>(c) 2026 OurShop</span>
      </footer>
    </main>
  </body>
</html>
"""

# Two objects for the two approaches:
soup = BeautifulSoup(HTML, "html.parser")   # for BeautifulSoup
tree = lxml_html.fromstring(HTML)           # for XPath (lxml)


def show(title, items):
    """Helper to print results neatly & make them easy to compare."""
    print(f"{title}  -> {len(items)} results")
    for it in items:
        print("   ", it)


print("soup & tree are ready. The example is used throughout the notebook ✅")


soup & tree are ready. The example is used throughout the notebook ✅


## 1. Select by tag (the easiest)

Grab all product names, which live in `<h2>` tags.

| | Syntax |
| --- | --- |
| **BS4** | `soup.find_all("h2")` |
| **XPath** | `//h2` (the element) or `//h2/text()` (its text directly) |


In [2]:
# BS4: grab the element, then its text
bs4_result = [h2.get_text(strip=True) for h2 in soup.find_all("h2")]
show("BS4  //h2", bs4_result)

# XPath: you can grab the text directly with /text()
xpath_result = [t.strip() for t in tree.xpath("//h2/text()")]
show("XPath //h2/text()", xpath_result)

assert bs4_result == xpath_result  # the results are the same


BS4  //h2  -> 4 results
    Learn Python
    Wireless Mouse
    Data Engineering 101
    Mechanical Keyboard
XPath //h2/text()  -> 4 results
    Learn Python
    Wireless Mouse
    Data Engineering 101
    Mechanical Keyboard


## 2. Select by id

An `id` is **unique** within a page, so it's the most reliable way to target a single element.

| | Syntax |
| --- | --- |
| **BS4** | `soup.find(id="content")` |
| **XPath** | `//*[@id="content"]` |


In [3]:
# BS4
header_bs4 = soup.find(id="top-nav")
print("BS4  :", header_bs4.name, "| role =", header_bs4.get("role"))

# XPath ([0] because xpath always returns a list)
header_xp = tree.xpath('//*[@id="top-nav"]')[0]
print("XPath:", header_xp.tag, "| role =", header_xp.get("role"))


BS4  : header | role = navigation
XPath: header | role = navigation


## 3. Select by class (1 token)

Grab all the prices, which live in `<span class="price">`.

| | Syntax |
| --- | --- |
| **BS4** | `soup.find_all(class_="price")` (note the underscore: `class_`) |
| **XPath** | `//span[@class="price"]` |

Here the class is just one word (`price`), so both are easy.
Be careful when the class has more than one word → see the next case.

In [4]:
# BS4: class_ with an underscore (because 'class' is a Python keyword)
price_bs4 = [s.get_text(strip=True) for s in soup.find_all(class_="price")]
show("BS4  class_='price'", price_bs4)

# XPath
price_xp = [s.text.strip() for s in tree.xpath('//span[@class="price"]')]
show("XPath //span[@class='price']", price_xp)

BS4  class_='price'  -> 4 results
    Rp75.000
    Rp150.000
    Rp120.000
    Rp350.000
XPath //span[@class='price']  -> 4 results
    Rp75.000
    Rp150.000
    Rp120.000
    Rp350.000


## 4. Multi-class — an important difference between BS4 & XPath ⚠️

Each product has a **combined** class, e.g.: `class="product card featured"`.
We want to grab every element that **has** the class `product`.

- **BS4** `class_="product"` → automatically matches if `product` is among
  the class words (token match). **Easy.**
- **XPath** `@class="product"` → matches only if the class is **exactly** `"product"`
  → here the result is **EMPTY** (because the value is `"product card featured"`).

The correct XPath solution for "contains the token `product`":

```
//*[contains(concat(' ', normalize-space(@class), ' '), ' product ')]
```

This adds spaces on both sides, then checks for `' product '` so it doesn't wrongly match
a word like `product-list`.

In [5]:
# BS4: token match -> 4 products right away
products_bs4 = soup.find_all(class_="product")
print("BS4  class_='product'      ->", len(products_bs4), "products")

# XPath WRONG: @class must be exact -> 0 results
wrong = tree.xpath('//div[@class="product"]')
print("XPath @class='product'     ->", len(wrong), "results (WRONG, empty)")

# XPath CORRECT: check the token with concat+contains
correct = tree.xpath(
    "//div[contains(concat(' ', normalize-space(@class), ' '), ' product ')]"
)
print("XPath contains(token)      ->", len(correct), "products (CORRECT)")

# Bonus: only 'featured' products
featured = tree.xpath(
    "//div[contains(concat(' ', normalize-space(@class), ' '), ' featured ')]/h2/text()"
)
show("XPath featured", [t.strip() for t in featured])

BS4  class_='product'      -> 4 products
XPath @class='product'     -> 0 results (WRONG, empty)
XPath contains(token)      -> 4 products (CORRECT)
XPath featured  -> 1 results
    Learn Python


## 5. Elements without a class → use position (nth)

The navigation menu `<li>` items have no class. We grab the **2nd menu link** ("Electronics").

- **BS4** has no "get the Nth element" inside `find`, so we grab all of them
  then **index the list** (remember: Python starts at 0).
- **XPath** has built-in positions: `[2]` (XPath starts at **1**).

| | Syntax |
| --- | --- |
| **BS4** | `soup.find("ul").find_all("li")[1]` |
| **XPath** | `//ul/li[2]/a` |

In [6]:
# BS4: index 1 (the 2nd element), 0-based
second_li = soup.find("ul").find_all("li")[1]
print("BS4  li[index 1]:", second_li.find("a").get_text(strip=True))

# XPath: li[2], 1-based
second_a = tree.xpath("//ul/li[2]/a/text()")[0]
print("XPath //ul/li[2]:", second_a.strip())

BS4  li[index 1]: Electronics
XPath //ul/li[2]: Electronics


## 6. Select by attribute (`data-*`)

Sometimes an element has no useful class/id, but it does have a `data-*` attribute.
Grab all products in the **books** category via `data-category="books"`.

| | Syntax |
| --- | --- |
| **BS4** | `soup.find_all(attrs={"data-category": "books"})` |
| **XPath** | `//*[@data-category="books"]` |

Grabbing an **attribute value** (not text): BS4 `el["data-id"]`, XPath `.../@data-id`.

In [7]:
# BS4: filter by attribute, then grab data-id + name
books_bs4 = soup.find_all(attrs={"data-category": "books"})
show("BS4  data-category=books", [(d["data-id"], d.find("h2").text) for d in books_bs4])

# XPath
books_xp = tree.xpath('//*[@data-category="books"]')
show("XPath data-category=books", [(d.get("data-id"), d.find("h2").text) for d in books_xp])

# Grab the list of data-id attribute values directly (all products) via XPath
ids = tree.xpath("//div[@data-id]/@data-id")
print("\nAll data-id (XPath //div[@data-id]/@data-id):", ids)

BS4  data-category=books  -> 2 results
    ('101', 'Learn Python')
    ('103', 'Data Engineering 101')
XPath data-category=books  -> 2 results
    ('101', 'Learn Python')
    ('103', 'Data Engineering 101')

All data-id (XPath //div[@data-id]/@data-id): ['101', '102', '103', '104']


## 7. aria-label & role (accessibility attributes)

`aria-label` and `role` are often lifesavers when an element has no class/id —
especially for buttons/icons. The approach is **exactly the same** as for regular attributes.

| Target | BS4 | XPath |
| --- | --- | --- |
| Element with `role="navigation"` | `soup.find(attrs={"role": "navigation"})` | `//*[@role="navigation"]` |
| Button with a specific aria-label | `soup.find(attrs={"aria-label": "..."})` | `//*[@aria-label="..."]` |

In [8]:
# role: find the navigation area
print("BS4  role=navigation :", soup.find(attrs={"role": "navigation"})["id"])
print("XPath role=navigation:", tree.xpath('//*[@role="navigation"]')[0].get("id"))

# aria-label: the 'add to cart' button for the Wireless Mouse
label = "Add Wireless Mouse to cart"
print("\nBS4  aria-label:", soup.find(attrs={"aria-label": label}).name)
print("XPath aria-label:", tree.xpath(f'//*[@aria-label="{label}"]')[0].tag)

# Grab all the Add button aria-labels (starts-with) -> see also case 9
labels = tree.xpath('//button/@aria-label')
show("\nAll <button> aria-labels", labels)

BS4  role=navigation : top-nav
XPath role=navigation: top-nav

BS4  aria-label: button
XPath aria-label: button

All <button> aria-labels  -> 3 results
    Add Learn Python to cart
    Add Wireless Mouse to cart
    Add Data Engineering 101 to cart


## 8. href — get links

Three common needs: **all links**, **internal product links** (start with `/product/`),
and **email links** (`mailto:`).

| Target | BS4 | XPath |
| --- | --- | --- |
| All hrefs | `soup.find_all("a", href=True)` | `//a/@href` |
| Start with `/product/` | `find_all("a", href=re.compile(r"^/product/"))` | `//a[starts-with(@href,"/product/")]/@href` |
| Email | `find_all("a", href=re.compile(r"^mailto:"))` | `//a[starts-with(@href,"mailto:")]/@href` |

In [9]:
# All hrefs
show("BS4  all hrefs", [a["href"] for a in soup.find_all("a", href=True)])
show("XPath //a/@href", tree.xpath("//a/@href"))

# Internal product links (start with /product/)
print()
show("BS4  ^/product/", [a["href"] for a in soup.find_all("a", href=re.compile(r"^/product/"))])
show("XPath starts-with", tree.xpath('//a[starts-with(@href,"/product/")]/@href'))

# Email links
print()
show("XPath mailto", tree.xpath('//a[starts-with(@href,"mailto:")]/@href'))

BS4  all hrefs  -> 9 results
    /
    /category/books
    /category/electronics
    https://promo.external.com/diskon
    /product/101
    /product/102
    /product/103
    /product/104
    mailto:cs@tokokita.id
XPath //a/@href  -> 9 results
    /
    /category/books
    /category/electronics
    https://promo.external.com/diskon
    /product/101
    /product/102
    /product/103
    /product/104
    mailto:cs@tokokita.id

BS4  ^/product/  -> 4 results
    /product/101
    /product/102
    /product/103
    /product/104
XPath starts-with  -> 4 results
    /product/101
    /product/102
    /product/103
    /product/104

XPath mailto  -> 1 results
    mailto:cs@tokokita.id


## 9. Partial matching (contains / starts-with / regex)

Sometimes we only know **part** of an attribute value. For example, all links whose class
contains `btn`, or hrefs that contain `category`.

- **BS4**: use a **regular expression** (`re.compile(...)`) on the attribute value.
- **XPath**: use the `contains(...)` or `starts-with(...)` functions.

> ⚠️ `contains(@class, "btn")` is **substring** matching, so it also matches
> `"btn-primary"`. To match an exact **token**, use the `concat` trick from case 4.

In [10]:
# All links whose class contains "btn"
btn_bs4 = soup.find_all("a", class_=re.compile("btn"))
show("BS4  class ~ btn (regex)", [a["class"] for a in btn_bs4])

btn_xp = tree.xpath('//a[contains(@class,"btn")]/@class')
show("XPath contains(@class,'btn')", btn_xp)

# hrefs that contain 'category'
print()
show("XPath contains href category", tree.xpath('//a[contains(@href,"category")]/@href'))

BS4  class ~ btn (regex)  -> 4 results
    ['btn', 'btn-primary']
    ['btn', 'btn-primary']
    ['btn', 'btn-primary']
    ['btn', 'btn-disabled']
XPath contains(@class,'btn')  -> 4 results
    btn btn-primary
    btn btn-primary
    btn btn-primary
    btn btn-disabled

XPath contains href category  -> 2 results
    /category/books
    /category/electronics


## 10. Select by text

Sometimes the most stable marker is the **text** itself. Example: find the badge reading "Out of stock".

| | Syntax |
| --- | --- |
| **BS4** (exact) | `soup.find("span", string="Out of stock")` |
| **BS4** (partial) | `soup.find_all("h2", string=re.compile("Engineering"))` |
| **XPath** (exact) | `//span[text()="Out of stock"]` |
| **XPath** (partial) | `//h2[contains(text(),"Engineering")]` |

In [11]:
# Exact text "Out of stock"
print("BS4  string='Out of stock' :", soup.find("span", string="Out of stock"))
print("XPath text()='Out of stock':", tree.xpath('//span[text()="Out of stock"]')[0].text)

# Partial text: product names that contain 'Engineering'
print()
hit_bs4 = [h.get_text(strip=True) for h in soup.find_all("h2", string=re.compile("Engineering"))]
show("BS4  regex 'Engineering'", hit_bs4)
hit_xp = [t.strip() for t in tree.xpath('//h2[contains(text(),"Engineering")]/text()')]
show("XPath contains 'Engineering'", hit_xp)

BS4  string='Out of stock' : <span class="badge">Out of stock</span>
XPath text()='Out of stock': Out of stock

BS4  regex 'Engineering'  -> 1 results
    Data Engineering 101
XPath contains 'Engineering'  -> 1 results
    Data Engineering 101


## 11. Navigation: parent / ancestor

A common pattern: find an easily identifiable element (e.g. the "Out of stock" badge),
then **go up** to its wrapping product to grab the `data-id`/name.

| | Syntax |
| --- | --- |
| **BS4** | `el.find_parent("div", class_="product")` |
| **XPath** | `el/ancestor::div[...token ' product '...]` |

> ⚠️ Watch out: `contains(@class,"product")` also matches `<div class="product-list">`
> (substring!). Use the `concat` token trick from case 4 so you go up to the correct product card.

In [12]:
# BS4: from the "Out of stock" badge -> go up to the product div
badge = soup.find("span", string="Out of stock")
out_of_stock_product = badge.find_parent("div", class_="product")
print("BS4  :", out_of_stock_product["data-id"], "-", out_of_stock_product.find("h2").text)

# XPath: from the badge node -> ancestor product div
# Use token-match (' product ') so it doesn't pick up <div class="product-list">
badge_xp = tree.xpath('//span[text()="Out of stock"]')[0]
product_xp = badge_xp.xpath(
    "ancestor::div[contains(concat(' ', normalize-space(@class), ' '), ' product ')]"
)[0]
print("XPath:", product_xp.get("data-id"), "-", product_xp.find("h2").text)

BS4  : 104 - Mechanical Keyboard
XPath: 104 - Mechanical Keyboard


## 12. Navigation: sibling (neighbor)

The "label then value next to it" pattern. Example: from the `<h2>` product name,
grab the `<span class="price">` that is its **next sibling**.

| | Syntax |
| --- | --- |
| **BS4** | `h2.find_next_sibling("span", class_="price")` |
| **XPath** | `//h2/following-sibling::span[@class="price"][1]` |

In [13]:
# BS4: from the first h2 -> the price next to it
h2 = soup.find("h2", class_="product-name")
price_sib = h2.find_next_sibling("span", class_="price")
print("BS4  :", h2.text, "->", price_sib.text)

# XPath: following-sibling, grab the first one [1]
pairs = tree.xpath(
    '//h2[@class="product-name"]/following-sibling::span[@class="price"][1]/text()'
)
show("XPath following-sibling (all products)", [p.strip() for p in pairs])

BS4  : Learn Python -> Rp75.000
XPath following-sibling (all products)  -> 4 results
    Rp75.000
    Rp150.000
    Rp120.000
    Rp350.000


## 13. Direct children only (`/` vs `//`)

`<main>` has a direct `<p>` ("Showing a selection of products...") **and** there's a `<p>`
inside `<footer>` (the email address). How do we grab **only the direct children**?

- **BS4**: `recursive=False` → only search at the direct-child level.
- **XPath**: `main/p` (one slash = direct child) vs `main//p` (two slashes = all descendants).

| | Syntax |
| --- | --- |
| **BS4** | `main.find_all("p", recursive=False)` |
| **XPath** | `//main/p` (direct) vs `//main//p` (all) |

In [14]:
main = soup.find("main")

# BS4: direct children only
direct = main.find_all("p", recursive=False)
show("BS4  recursive=False (direct children)", [p.get_text(strip=True) for p in direct])

# BS4: all p (including in the footer)
all_p = main.find_all("p")
show("BS4  all p (default)", [p.get_text(strip=True) for p in all_p])

# XPath: / vs //
print()
show("XPath //main/p (direct)", [t.strip() for t in tree.xpath("//main/p/text()")])
show("XPath //main//p (all)", [t.strip() for t in tree.xpath("//main//p//text()")])

BS4  recursive=False (direct children)  -> 1 results
    Showing a selection of featured products.
BS4  all p (default)  -> 2 results
    Showing a selection of featured products.
    Contact:cs@tokokita.id

XPath //main/p (direct)  -> 1 results
    Showing a selection of featured products.
XPath //main//p (all)  -> 3 results
    Showing a selection of featured products.
    Contact:
    cs@tokokita.id


## 14. Combined conditions + numeric filters

Combine several conditions with `and`/`or` in XPath. Here XPath has an advantage:
it can **filter numbers** directly with `number(...)`.

Example: find products with a **price > 200,000** (data-price), then grab their names.

- **BS4**: there's no numeric operator in `find` → we **filter in Python**.
- **XPath**: `//span[@class="price"][number(@data-price) > 200000]`.

Plus an `and` example: links whose **class is btn-primary** AND **href starts with `/product/10`**.

In [15]:
# Price > 200,000
# BS4: filter manually in Python
expensive_bs4 = [
    s.find_parent("div").find("h2").text
    for s in soup.find_all("span", class_="price")
    if int(s["data-price"]) > 200000
]
show("BS4  price>200000 (Python filter)", expensive_bs4)

# XPath: inline numeric filter + go up to the preceding h2 sibling
expensive_xp = tree.xpath(
    '//span[@class="price"][number(@data-price) > 200000]'
    '/preceding-sibling::h2/text()'
)
show("XPath number(@data-price)>200000", [t.strip() for t in expensive_xp])

# AND combination: class btn-primary AND href starts with /product/10
print()
combined = tree.xpath(
    '//a[contains(@class,"btn-primary") and starts-with(@href,"/product/10")]/@href'
)
show("XPath AND (btn-primary & /product/10)", combined)

BS4  price>200000 (Python filter)  -> 1 results
    Mechanical Keyboard
XPath number(@data-price)>200000  -> 1 results
    Mechanical Keyboard

XPath AND (btn-primary & /product/10)  -> 3 results
    /product/101
    /product/102
    /product/103


## 15. Mini-project: extract all products → list of dicts

Combine every technique: for each product, grab the **name, category, price (number),
link, and stock status**. We write a BS4 version and an XPath version, then make sure the results match.

Note the handling of **incomplete** data (the "Mechanical Keyboard" product has no
rating, but it does have an "Out of stock" badge).

In [16]:
import pandas as pd

PRODUCT_XPATH = "//div[contains(concat(' ', normalize-space(@class), ' '), ' product ')]"


def parse_bs4():
    result = []
    for p in soup.find_all("div", class_="product"):
        badge = p.find("span", class_="badge")
        result.append(
            {
                "name": p.find("h2", class_="product-name").get_text(strip=True),
                "category": p["data-category"],
                "price": int(p.find("span", class_="price")["data-price"]),
                "link": p.find("a", class_="btn")["href"],
                "stock": "out_of_stock" if badge and badge.get_text(strip=True) == "Out of stock" else "in_stock",
            }
        )
    return result


def parse_xpath():
    result = []
    for p in tree.xpath(PRODUCT_XPATH):
        badge = p.xpath('.//span[@class="badge"]/text()')  # relative: starts with "."
        result.append(
            {
                "name": p.xpath('.//h2[@class="product-name"]/text()')[0].strip(),
                "category": p.get("data-category"),
                "price": int(p.xpath('.//span[@class="price"]/@data-price')[0]),
                "link": p.xpath('.//a[contains(@class,"btn")]/@href')[0],
                "stock": "out_of_stock" if badge and badge[0].strip() == "Out of stock" else "in_stock",
            }
        )
    return result


data_bs4 = parse_bs4()
data_xp = parse_xpath()
assert data_bs4 == data_xp, "BS4 and XPath results must match!"
print("BS4 == XPath ✅  (identical results)\n")
pd.DataFrame(data_bs4)

BS4 == XPath ✅  (identical results)



,name,category,price,link,stock
0,Learn Python,books,75000,/product/101,in_stock
1,Wireless Mouse,electronics,150000,/product/102,in_stock
2,Data Engineering 101,books,120000,/product/103,in_stock
3,Mechanical Keyboard,electronics,350000,/product/104,out_of_stock


## 16. Cheat Sheet — BeautifulSoup ↔ XPath

| Goal | BeautifulSoup | XPath (lxml / Selenium) |
| --- | --- | --- |
| All `h2` tags | `soup.find_all("h2")` | `//h2` |
| By id | `soup.find(id="x")` | `//*[@id="x"]` |
| Class (1 token) | `soup.find_all(class_="price")` | `//*[@class="price"]` |
| Class (multi-class) | `soup.find_all(class_="product")` | `//*[contains(concat(' ',normalize-space(@class),' '),' product ')]` |
| Nth element | `soup.find_all("li")[1]` (0-based) | `(//li)[2]` (1-based) |
| Exact attribute | `soup.find_all(attrs={"data-id":"101"})` | `//*[@data-id="101"]` |
| aria / role | `soup.find(attrs={"role":"navigation"})` | `//*[@role="navigation"]` |
| Get attribute value | `el["href"]` | `.../@href` |
| Starts with | `re.compile(r"^/product/")` | `starts-with(@href,"/product/")` |
| Contains | `re.compile("btn")` | `contains(@class,"btn")` |
| Exact text | `soup.find(string="Out of stock")` | `//*[text()="Out of stock"]` |
| Partial text | `string=re.compile("Eng")` | `contains(text(),"Eng")` |
| Parent / ancestor | `el.find_parent("div")` | `ancestor::div` |
| Next sibling | `el.find_next_sibling("span")` | `following-sibling::span[1]` |
| Direct child | `el.find_all("p", recursive=False)` | `el/p` (vs `el//p`) |
| Combine conditions | (filter in Python) | `[a and b]`, `number(@x) > n` |

## Pitfalls (common headaches)

- **XPath `@class="..."` must be EXACT.** For multi-class use the `concat`/`contains` trick.
- **Different starting index:** Python list `[0]`, XPath position `[1]`.
- **`find()` can be `None`.** Accessing `.text`/`["href"]` on `None` → error. Check first (see the optional badge).
- **`contains()` is substring.** `contains(@class,"btn")` also matches `"btn-primary"`; if you need an exact token use `concat`.
- **Relative XPath** from an element must start with a dot: `el.xpath(".//span")`. Without the dot (`//span`) it searches the **entire document**.
- **`/text()` vs `.text`:** XPath `/text()` returns a string; if there are child tags, the text can be cut off (use `.text_content()` in lxml for the combined text).

## Exercises

1. Get **all category links** (`/category/...`) — write a BS4 version **and** an XPath version.
2. Get the **name of the most expensive product** using numeric XPath (`number(@data-price)`).
3. From the text `"(c) 2026 OurShop"` in the footer, grab its `<span>` element via text matching.

Try it yourself before looking at the example answer below.

In [17]:
# --- Example answers ---

# 1. All category links
show("BS4  category", [a["href"] for a in soup.find_all("a", href=re.compile(r"^/category/"))])
show("XPath category", tree.xpath('//a[starts-with(@href,"/category/")]/@href'))

# 2. Name of the most expensive product (pure XPath: "no other price is greater")
print()
most_expensive = tree.xpath(
    '//span[@class="price"]'
    '[not(number(@data-price) < number(//span[@class="price"]/@data-price))]'
    '/preceding-sibling::h2/text()'
)
print("Most expensive product (XPath):", [t.strip() for t in most_expensive])

# 3. The copyright span via text matching
print()
print("BS4  :", soup.find("span", string=re.compile("OurShop")).get_text(strip=True))
print("XPath:", tree.xpath('//footer//span[contains(text(),"OurShop")]/text()')[0].strip())

BS4  category  -> 2 results
    /category/books
    /category/electronics
XPath category  -> 2 results
    /category/books
    /category/electronics

Most expensive product (XPath): ['Learn Python', 'Wireless Mouse', 'Data Engineering 101', 'Mechanical Keyboard']

BS4  : (c) 2026 OurShop
XPath: (c) 2026 OurShop
